# Finetune Notebook (SPR Probing)

"
"This notebook runs end-to-end finetuning with `probing.finetune` using the default mapping:
"
"- entailment: `label >= 4`
"
"- not entailment: `label <= 3`
"
"- drop inapplicable rows (`applicable == false`)

"
"Use `QUICK_RUN = True` for a fast smoke test, then switch to `False` for full training.

In [1]:
import json
import shlex
import subprocess
import sys
import zipfile
from collections import Counter
from pathlib import Path

ROOT = Path(".").resolve()
if not (ROOT / "probing").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

Project root: /home/tubinex/Documents/Projects/Semantik


In [2]:
# Core configuration
MODEL_NAME        = "roberta-large-mnli"
HYPOTHESIS_METHOD = "type-aware-templates" 
DEVICE            = "auto"

PAIRS_JSONL = ROOT / "data" / "hypotheses" / HYPOTHESIS_METHOD / "pairs.jsonl"
PAIRS_ZIP   = PAIRS_JSONL.with_suffix(".zip")

OUTPUT_DIR = ROOT / "artifacts" / "finetune" / HYPOTHESIS_METHOD / f"{MODEL_NAME}-spr-bin"

# Quick smoke test vs full run
QUICK_RUN = False
SMOKE_MAX_TRAIN = 2000
SMOKE_MAX_EVAL = 500
SMOKE_EPOCHS = 1.0

# Full-run defaults
FULL_EPOCHS = 3.0
TRAIN_BATCH = 16
EVAL_BATCH = 32
MAX_LEN = 256
LR = 2e-5

print("Model:", MODEL_NAME)
print("Hypothesis method:", HYPOTHESIS_METHOD)
print("Pairs JSONL:", PAIRS_JSONL)
print("Output:", OUTPUT_DIR)
print("QUICK_RUN:", QUICK_RUN)

Model: roberta-large-mnli
Hypothesis method: type-aware-templates
Pairs JSONL: /home/tubinex/Documents/Projects/Semantik/data/hypotheses/type-aware-templates/pairs.jsonl
Output: /home/tubinex/Documents/Projects/Semantik/artifacts/finetune/type-aware-templates/roberta-large-mnli-spr-bin
QUICK_RUN: False


In [3]:
if not PAIRS_JSONL.exists():
    if not PAIRS_ZIP.exists():
        raise FileNotFoundError(f"Missing input zip: {PAIRS_ZIP}")

    with zipfile.ZipFile(PAIRS_ZIP) as zf:
        members = zf.namelist()
        jsonl_members = [m for m in members if m.endswith(".jsonl")]
        if not jsonl_members:
            raise RuntimeError(f"No .jsonl file found in {PAIRS_ZIP}")
        member = jsonl_members[0]
        PAIRS_JSONL.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(member) as src, PAIRS_JSONL.open("wb") as dst:
            dst.write(src.read())

print(f"Ready: {PAIRS_JSONL} ({PAIRS_JSONL.stat().st_size / (1024**2):.1f} MB)")

Ready: /home/tubinex/Documents/Projects/Semantik/data/hypotheses/type-aware-templates/pairs.jsonl (83.8 MB)


In [4]:
split_counts = Counter()
mapped_counts = Counter()
raw_label_counts = Counter()
rows_total = 0
rows_kept = 0

with PAIRS_JSONL.open(encoding="utf-8") as fh:
    for line in fh:
        rec = json.loads(line)
        rows_total += 1

        split = rec.get("split", "")
        split_counts[split] += 1

        label = int(rec["label"])
        raw_label_counts[label] += 1

        applicable = rec.get("applicable", True)
        if isinstance(applicable, str):
            applicable = applicable.strip().lower() == "true"

        if not applicable:
            continue

        rows_kept += 1
        mapped_counts["entailment" if label >= 4 else "not_entailment"] += 1

print("Total rows:", rows_total)
print("Rows kept (applicable only):", rows_kept)
print("Split counts:", dict(split_counts))
print("Raw label counts:", dict(sorted(raw_label_counts.items())))
print("Mapped label counts:", dict(mapped_counts))

Total rows: 175284
Rows kept (applicable only): 95845
Split counts: {'train': 136998, 'test': 19008, 'dev': 19278}
Raw label counts: {1: 100272, 2: 1963, 3: 17909, 4: 2627, 5: 52513}
Mapped label counts: {'not_entailment': 40705, 'entailment': 55140}


In [5]:
import ast
import re
import subprocess
import ipywidgets as widgets
from IPython.display import display

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-m", "probing.finetune",
    "--model", MODEL_NAME,
    "--input", str(PAIRS_JSONL),
    "--output-dir", str(OUTPUT_DIR),
    "--train-split", "train",
    "--eval-split", "dev",
    "--label-threshold", "4",
    "--max-length", str(MAX_LEN),
    "--per-device-train-batch-size", str(TRAIN_BATCH),
    "--per-device-eval-batch-size", str(EVAL_BATCH),
    "--learning-rate", str(LR),
    "--device", DEVICE,
]
if QUICK_RUN:
    cmd += ["--num-train-epochs", str(SMOKE_EPOCHS),
            "--max-train-examples", str(SMOKE_MAX_TRAIN),
            "--max-eval-examples", str(SMOKE_MAX_EVAL)]
else:
    cmd += ["--num-train-epochs", str(FULL_EPOCHS)]

_W = "98%"

status_label = widgets.HTML("<b>Status:</b> starting…")

train_bar = widgets.IntProgress(
    value=0, min=0, max=1,
    description="Train", bar_style="info",
    layout=widgets.Layout(width=_W, height="20px"),
)
train_label = widgets.Label("step 0 / ?")

eval_bar = widgets.IntProgress(
    value=0, min=0, max=1,
    description="Eval", bar_style="info",
    layout=widgets.Layout(width=_W, height="20px"),
)
eval_label = widgets.Label("waiting…")

metrics_html = widgets.HTML("")

display(widgets.VBox([
    status_label,
    train_bar,
    eval_bar,
    train_label,
    eval_label,
    metrics_html,
], layout=widgets.Layout(width="100%")))

_RE_TOTAL   = re.compile(r"Total optimization steps\s*=\s*(\d+)")
_RE_TQDM    = re.compile(r"^\s*\S.*\d+%\|")
_RE_TQDM_TR = re.compile(
    r"^\s*\d+%\|.*\|\s*(\d+)/(\d+)\s*\[(\d+:\d+)<(\d+:\d+),\s*([\d.]+)it/s"
)

def _try_parse_dict(line):
    s = line.strip()
    if s.startswith("{") and s.endswith("}"):
        try:
            return ast.literal_eval(s)
        except Exception:
            pass
    return None

def _floatify(d):
    return {k: float(v) if isinstance(v, (int, float, str)) else v for k, v in d.items()}

total_steps = None
in_eval     = False
last_train_metrics = {}
last_eval_metrics  = {}

proc = subprocess.Popen(
    cmd, cwd=ROOT,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

for raw in proc.stdout:
    line = raw.rstrip("\n")

    if "Running Evaluation" in line or "***** Running Evaluation" in line:
        in_eval = True
        eval_bar.bar_style = "info"
        eval_label.value = "running…"

    m = _RE_TOTAL.search(line)
    if m:
        total_steps = int(m.group(1))
        train_bar.max = total_steps
        status_label.value = "<b>Status:</b> training"
        in_eval = False

    m = _RE_TQDM_TR.match(line)
    if m:
        current, total = int(m.group(1)), int(m.group(2))
        elapsed, remaining, rate = m.group(3), m.group(4), float(m.group(5))
        if in_eval:
            eval_bar.max   = total
            eval_bar.value = current
            eval_label.value = (
                f"step {current} / {total}"
                f"  |  elapsed {elapsed}  |  ETA {remaining}  |  {rate:.2f} it/s"
            )
        else:
            if total_steps is None:
                train_bar.max = total
            train_bar.value = current
            train_label.value = (
                f"step {current} / {total_steps or total}"
                f"  |  elapsed {elapsed}  |  ETA {remaining}  |  {rate:.2f} it/s"
            )
        continue

    if _RE_TQDM.match(line):
        continue

    d = _try_parse_dict(line)

    if d and "loss" in d and "eval_loss" not in d:
        d = _floatify(d)
        last_train_metrics = d
        epoch = d.get("epoch", 0)
        step = int(round(epoch * (total_steps or 1)))
        train_bar.value = step
        metrics_html.value = (
            f"<b>Train →</b> loss: {d['loss']:.4f} &nbsp;|&nbsp; "
            f"lr: {d.get('learning_rate', 0):.2e} &nbsp;|&nbsp; "
            f"epoch: {epoch:.2f}"
            + (f"&emsp;<b>Eval →</b> loss: {last_eval_metrics['eval_loss']:.4f} &nbsp;|&nbsp; "
               f"f1: {last_eval_metrics.get('eval_f1', 0):.4f} &nbsp;|&nbsp; "
               f"acc: {last_eval_metrics.get('eval_accuracy', 0):.4f}"
               if last_eval_metrics else "")
        )
        continue

    if d and "eval_loss" in d:
        d = _floatify(d)
        last_eval_metrics = d
        in_eval = False
        eval_bar.bar_style = "success"
        eval_bar.value = eval_bar.max
        eval_label.value = (
            f"loss: {d['eval_loss']:.4f}  |  "
            f"f1: {d.get('eval_f1', 0):.4f}  |  "
            f"acc: {d.get('eval_accuracy', 0):.4f}"
        )
        metrics_html.value = (
            (f"<b>Train →</b> loss: {last_train_metrics['loss']:.4f} &nbsp;|&nbsp; "
             f"lr: {last_train_metrics.get('learning_rate', 0):.2e} &nbsp;|&nbsp; "
             f"epoch: {last_train_metrics.get('epoch', 0):.2f}&emsp;"
             if last_train_metrics else "")
            + f"<b>Eval →</b> loss: {d['eval_loss']:.4f} &nbsp;|&nbsp; "
              f"f1: {d.get('eval_f1', 0):.4f} &nbsp;|&nbsp; "
              f"acc: {d.get('eval_accuracy', 0):.4f}"
        )
        continue

proc.wait()
if proc.returncode != 0:
    status_label.value = "<b>Status:</b> <span style='color:red'>FAILED</span>"
    raise subprocess.CalledProcessError(proc.returncode, cmd)

train_bar.bar_style = "success"
train_bar.value = train_bar.max
status_label.value = "<b>Status:</b> <span style='color:green'>done ✓</span>"

In [6]:
meta_path = OUTPUT_DIR / "finetune_metadata.json"
if not meta_path.exists():
    raise FileNotFoundError(f"Missing metadata file: {meta_path}")

meta = json.loads(meta_path.read_text(encoding="utf-8"))

print("Saved model:", OUTPUT_DIR)
print("Train examples:", meta["data"]["train_examples"])
print("Eval examples:", meta["data"]["eval_examples"])
print("Train label counts:", meta["data"]["train_label_counts"])
print("Eval label counts:", meta["data"]["eval_label_counts"])

print("Metrics:")
for k, v in sorted(meta.get("metrics", {}).items()):
    if isinstance(v, (int, float)):
        print(f"- {k}: {v:.6f}")
    else:
        print(f"- {k}: {v}")

Saved model: /home/tubinex/Documents/Projects/Semantik/artifacts/finetune/type-aware-templates/roberta-large-mnli-spr-bin
Train examples: 77574
Eval examples: 10316
Train label counts: {'not_entailment': 32106, 'entailment': 45468}
Eval label counts: {'not_entailment': 4358, 'entailment': 5958}
Metrics:
- epoch: 3.000000
- eval_accuracy: 0.900931
- eval_f1: 0.914147
- eval_loss: 0.424559
- eval_precision: 0.915069
- eval_recall: 0.913226
- eval_runtime: 16.289000
- eval_samples_per_second: 633.311000
- eval_steps_per_second: 19.829000


In [7]:
from probing.prober import Prober

sample = None
with PAIRS_JSONL.open(encoding="utf-8") as fh:
    for line in fh:
        rec = json.loads(line)
        if rec.get("split") == "dev":
            applicable = rec.get("applicable", True)
            if isinstance(applicable, str):
                applicable = applicable.strip().lower() == "true"
            if applicable:
                sample = rec
                break

if sample is None:
    raise RuntimeError("No dev/applicable sample found")

ft_prober = Prober(str(OUTPUT_DIR), device=DEVICE, threshold=0.5)
pred = ft_prober.predict_one(sample["target_text"], sample["hypothesis"], return_inputs=True)

print("Sample property:", sample.get("property"))
print("Gold label:", sample.get("label"), "(mapped entailment if >=4)")
print("Prediction:", pred["pred_label"])
print("p_entail:", pred["p_entail"])
print("p_not_entail:", pred["p_not_entail"])

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Sample property: awareness
Gold label: 5 (mapped entailment if >=4)
Prediction: ENTAILS
p_entail: 0.9989022
p_not_entail: 0.00109772
